# 5. DynamoDB Merchant Lookup + Bedrock Tool Call Experiment

Extends existing DynamoDB setup with a Bedrock tool call experiment.

**Sections:**
1. Imports and setup (existing)
2. DynamoDB client and `lookup_merchant` (existing)
3. Existing lookup tests (existing)
4. Bedrock tool call experiment (NEW)
   - 4a. Tool schema definition
   - 4b. Bedrock client with tool loop
   - 4c. Test transactions
   - 4d. Latency and cost comparison

---
## 1. Imports and Setup

In [ ]:
import boto3
import json
import time
import os
import gzip
from dotenv import load_dotenv

REGION   = 'ap-southeast-2'
MODEL_ID = 'anthropic.claude-3-5-sonnet-20241022-v2:0'

load_dotenv(dotenv_path=r'/home/sagemaker-user/swtest1/llm_poc/.env', override=True)
DEV_ROLE = os.getenv('DEV_ROLE')
print(f'DEV_ROLE: {DEV_ROLE}')

for key in ['AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY', 'AWS_SESSION_TOKEN', 'AWS_BEARER_TOKEN_BEDROCK']:
    os.environ.pop(key, None)

---
## 2. DynamoDB Client and lookup_merchant

In [ ]:
#bedrock = boto3.client('bedrock-runtime', region_name=REGION)

sts_client  = boto3.client('sts')
sts_session = sts_client.assume_role(
    RoleArn=DEV_ROLE,
    RoleSessionName='notebook-session'
)
creds = sts_session['Credentials']

dynamodb_client = boto3.client(
    'dynamodb',
    region_name=REGION,
    aws_access_key_id=creds['AccessKeyId'],
    aws_secret_access_key=creds['SecretAccessKey'],
    aws_session_token=creds['SessionToken'],
)



def lookup_merchant(merchant_name: str) -> dict:
    normalised = merchant_name.upper().strip()
    escaped    = normalised.translate(str.maketrans({"'": r"''"}))
    try:
        resp  = dynamodb_client.execute_statement(
            Statement=(
                "SELECT * FROM mapping "
                "WHERE \"type\" = 'merchant' "
                f"AND \"entity\" = '{escaped}'"
            )
        )
        items = resp.get('Items', [])
        if items:
            item = items[0]
            def unwrap(v):
                if isinstance(v, dict) and v:
                    return next(iter(v.values()))
                return None
            return {
                'found':    True,
                'entity':   unwrap(item.get('entity')),
                'category': unwrap(item.get('category')),
                'mapping':  unwrap(item.get('mapping')),
                'priority': unwrap(item.get('priority')),
            }
        return {'found': False, 'entity': normalised}
    except Exception as e:
        return {'found': False, 'error': str(e)}


print('DynamoDB client and lookup_merchant ready')

---
## 3. Existing Lookup Tests

In [ ]:
# Known merchant tests
test_merchants = ['COLES', 'WOOLWORTHS', 'NETFLIX', 'UBER', 'BUNNINGS', 'MEDICARE', 'ATO']
for m in test_merchants:
    print(f'{m}: {lookup_merchant(m)}')

In [ ]:
# Sample raw entities from table
resp = dynamodb_client.execute_statement(
    Statement="SELECT * FROM mapping WHERE \"type\" = 'merchant'"
)
for item in resp.get('Items', [])[:10]:
    print({k: list(v.values())[0] for k, v in item.items()})

---
## 4. Bedrock Tool Call Experiment

Tests whether Claude can extract a merchant name from a raw transaction description and call `lookup_merchant` as a tool.

**Flow:**
1. Send transaction description + amount to Claude with `lookup_merchant` defined as a tool
2. Claude extracts merchant, calls tool
3. We execute the actual DynamoDB lookup and return result
4. Claude uses result to produce final `base_category_key`

This is Pass 2 of the two-pass merchant extraction design.

### 4a. Tool Schema

In [ ]:
# Tool definition - passed to Bedrock in every request
MERCHANT_LOOKUP_TOOL = {
    'name': 'lookup_merchant',
    'description': (
        'Look up a merchant name in the category mapping database. '
        'Returns the base_category_key if the merchant is found, or not_found if unknown. '
        'Always normalise the merchant name before calling: remove location suffixes, '
        'legal entity suffixes, and card numbers. Use the core brand name only. '
        'Examples: "WOOLWORTHS METRO SYDNEY" -> "WOOLWORTHS"; '
        '"NETFLIX.COM/AU" -> "NETFLIX"; "AMZNPRIMEAU MEMBERSHIP" -> "AMAZON PRIME"'
    ),
    'input_schema': {
        'type': 'object',
        'properties': {
            'merchant_name': {
                'type': 'string',
                'description': 'Normalised merchant name in UPPER CASE e.g. WOOLWORTHS, NETFLIX, UBER'
            }
        },
        'required': ['merchant_name']
    }
}

print('Tool schema defined')
print(json.dumps(MERCHANT_LOOKUP_TOOL, indent=2))

### 4b. Bedrock Client with Tool Loop

In [ ]:
#bedrock = boto3.client('bedrock-runtime', region_name=REGION)

TOOL_SYSTEM_PROMPT = """You are a financial transaction categorisation engine.
Given a transaction description and amount, you must:
1. Identify the merchant from the description
2. Call lookup_merchant to find the category
3. Return ONLY a JSON object: {"merchant": "Name", "base_category_key": "KEY", "source": "db_lookup | llm_fallback", "confidence": 8}

If lookup_merchant returns not_found, use your own knowledge to assign the most likely category.
Set source to 'db_lookup' if the category came from the tool, 'llm_fallback' if not.
Return ONLY the JSON object. No explanation, no text before or after.
"""


def call_bedrock_with_tools(user_prompt: str, max_retries: int = 3) -> dict:
    """
    Bedrock call with tool use loop.
    Handles the two-turn conversation: initial call -> tool execution -> follow-up call.
    Returns dict with final text response + metadata.
    """
    bedrock = boto3.client('bedrock-runtime', region_name='ap-southeast-2')
    messages   = [{'role': 'user', 'content': user_prompt}]
    tool_calls = 0
    t_start    = time.time()
    input_tokens = output_tokens = 0

    for attempt in range(max_retries):
        try:
            body = json.dumps({
                'anthropic_version': 'bedrock-2023-05-31',
                'max_tokens': 500,
                'temperature': 0.0,
                'system': TOOL_SYSTEM_PROMPT,
                'tools': [MERCHANT_LOOKUP_TOOL],
                'messages': messages,
            })
            resp   = bedrock.invoke_model(
                modelId=MODEL_ID,
                contentType='application/json',
                accept='application/json',
                body=body,
            )
            data   = json.loads(resp['body'].read())
            input_tokens  += data.get('usage', {}).get('input_tokens', 0)
            output_tokens += data.get('usage', {}).get('output_tokens', 0)
            stop_reason    = data.get('stop_reason')
            content        = data.get('content', [])

            # Tool use requested
            if stop_reason == 'tool_use':
                tool_calls += 1
                # Add assistant response to messages
                messages.append({'role': 'assistant', 'content': content})

                # Execute all tool calls in the response
                tool_results = []
                for block in content:
                    if block.get('type') == 'tool_use':
                        tool_input  = block.get('input', {})
                        merchant    = tool_input.get('merchant_name', '')
                        db_result   = lookup_merchant(merchant)
                        result_text = (
                            f'found: {db_result["category"]}'
                            if db_result.get('found')
                            else 'not_found'
                        )
                        print(f'  Tool call: lookup_merchant("{merchant}") -> {result_text}')
                        tool_results.append({
                            'type':       'tool_result',
                            'tool_use_id': block['id'],
                            'content':    result_text,
                        })

                # Add tool results and loop
                messages.append({'role': 'user', 'content': tool_results})
                continue  # loop back for follow-up call

            # Final text response
            text = next(
                (b.get('text', '') for b in content if b.get('type') == 'text'), ''
            )
            elapsed = time.time() - t_start
            # Cost: Sonnet v2 $3/M input, $15/M output
            cost = (input_tokens / 1_000_000 * 3.0) + (output_tokens / 1_000_000 * 15.0)
            return {
                'text':          text,
                'tool_calls':    tool_calls,
                'input_tokens':  input_tokens,
                'output_tokens': output_tokens,
                'cost':          cost,
                'elapsed_s':     round(elapsed, 2),
            }

        except Exception as e:
            if 'ThrottlingException' in str(type(e)) and attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            else:
                raise


print('Bedrock tool call client ready')

### 4c. Test Transactions

Four scenarios:
- **A:** Clean merchant name in description (should extract and find in DB)
- **B:** Noisy description, merchant embedded (tests LLM extraction quality)
- **C:** Non-spend / government payment (should not find merchant, fall back to LLM)
- **D:** No match in DB (tests llm_fallback path)

In [ ]:
test_transactions = [
    {
        'label':  'A - Clean merchant',
        'desc':   'WOOLWORTHS 3082 SYDNEY NSW AUS',
        'amount': -87.40,
    },
    {
        'label':  'B - Noisy description',
        'desc':   'AMZNPRIMEAU MEMBERSHIP SYDNEY SOUTH NS AUS, Card xx9246, Value date: 09/10/2025',
        'amount': -9.99,
    },
    {
        'label':  'C - Non-spend / government',
        'desc':   'CENTRELINK PAYMENT 123456789',
        'amount': 980.00,
    },
    {
        'label':  'D - No DB match expected',
        'desc':   'TRANSFER TO LINKED ACCOUNT 001234567',
        'amount': -500.00,
    },
]


def build_tool_user_prompt(desc: str, amount: float) -> str:
    return f'description: {desc}\namount: {amount}'


results = []
for txn in test_transactions:
    print(f"\n{'='*60}")
    print(f"Test {txn['label']}")
    print(f"Description: {txn['desc']}")
    print(f"Amount: {txn['amount']}")
    prompt = build_tool_user_prompt(txn['desc'], txn['amount'])
    res    = call_bedrock_with_tools(prompt)
    print(f'Response: {res["text"]}')
    print(f'Tool calls: {res["tool_calls"]} | Tokens: {res["input_tokens"]}in/{res["output_tokens"]}out | Cost: ${res["cost"]:.5f} | Time: {res["elapsed_s"]}s')
    results.append({**txn, **res})

In [ ]:
import boto3
bedrock = boto3.client("bedrock-runtime", region_name=REGION)
print("Bedrock client ready")

### 4d. Latency and Cost Summary

In [ ]:
import pandas as pd

df = pd.DataFrame(results)[['label', 'tool_calls', 'input_tokens', 'output_tokens', 'cost', 'elapsed_s', 'text']]
print(df.to_string(index=False))

print(f'\nAvg latency (tool call):    {df[df["tool_calls"]>0]["elapsed_s"].mean():.2f}s')
print(f'Avg latency (no tool call): {df[df["tool_calls"]==0]["elapsed_s"].mean():.2f}s')
print(f'Avg cost per row:           ${df["cost"].mean():.5f}')
print(f'\nNote: Run 3 baseline is ~$0.058/row at ~3.3s/row (with naive call)')
print(f'Tool call adds ~1 extra Bedrock call for rows where merchant is found')